# 🧪 W4-D2 概念实验：余弦、欧氏、点积到底差在哪？

> 配套阅读：`ima/第4周-Day2-向量检索与相似度计算.md`（Embedding 原理、三种相似度推导在那边）
>
> 这个 notebook 用手算级的小实验回答三个问题：
> 1. 同一组向量，**点积 / 余弦 / 欧氏距离**给出的排序什么时候一致、什么时候打架？
> 2. 为什么语义检索选余弦，而不是关键词匹配？
> 3. 归一化之后，余弦和欧氏距离为什么会变成"一回事"？
>
> 实验环境：纯 numpy 手工构造向量（模拟 Embedding 语义空间），无任何模型调用。

## 实验 1：三种相似度手算对比 + "幅度陷阱"

用 4 维"语义向量"模拟产品 Embedding（维度含义：`[甜品感, 水果感, 冰爽感, 温热感]`）。
关键观察：**把"芒果冰沙"的向量放大 3 倍（语义完全没变）**，余弦说"还是它"，欧氏距离和点积却翻脸。

In [ ]:
import numpy as np
np.set_printoptions(precision=2, suppress=True)

# 维度含义：[甜品感, 水果感, 冰爽感, 温热感]
vecs = {
    "芒果冰沙": np.array([0.9, 0.9, 0.9, 0.0]),
    "杨枝甘露": np.array([0.8, 0.8, 0.7, 0.0]),
    "热红豆沙": np.array([0.8, 0.0, 0.0, 0.9]),
    "草莓圣代": np.array([0.85, 0.85, 0.3, 0.0]),
}
query = np.array([0.7, 0.85, 0.95, 0.0])   # 顾客心里想的是"夏天冰爽的水果甜品"

def cos(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

def eucl(a, b):
    return np.linalg.norm(a - b)

print("查询意图向量 =", query, "（冰爽+水果）\n")
print(f"{'产品':　<6}{'点积':>8}{'余弦':>8}{'欧氏距离':>10}")
for name, v in sorted(vecs.items(), key=lambda kv: -cos(kv[1], query)):
    print(f"{name:　<6}{query @ v:>8.3f}{cos(query, v):>8.3f}{eucl(query, v):>10.3f}")

# ---- 幅度陷阱：语义不变、只把"芒果冰沙"的向量放大 3 倍 ----
print("\n把「芒果冰沙」向量 x3（语义完全没变，只是'音量'变大）：")
big = vecs["芒果冰沙"] * 3
print(f"  余弦     : {cos(query, vecs['芒果冰沙']):.3f} → {cos(query, big):.3f}  （纹丝不动 ✓）")
print(f"  欧氏距离 : {eucl(query, vecs['芒果冰沙']):.3f} → {eucl(query, big):.3f}  （突然变'远' ✗）")
print(f"  点积     : {query @ vecs['芒果冰沙']:.3f} → {query @ big:.3f}  （突然变'大'，会挤掉别的产品 ✗）")

# 放大后的排序变化：点积被幅度绑架
print("\n按点积排序（放大版参战后）：")
for name, v in sorted({**vecs, "芒果冰沙x3": big}.items(), key=lambda kv: -query @ kv[1]):
    print(f"  {name}: 点积={query @ v:.2f}  余弦={cos(query, v):.3f}")
print("→ 点积排名被幅度绑架；余弦只看方向（语义），这才是语义检索要的度量。")

## 实验 2：语义检索 vs 关键词匹配 —— "解暑的甜品"查什么？

顾客说"解暑的甜品"，字面上和"芒果冰沙""杨枝甘露"**没有一个字相同**。
关键词匹配（字符重合度）全军覆没，语义向量却能把"冷饮系"产品排到前面。

In [ ]:
import numpy as np

# 手工"Embedding"：维度 = [冰爽解暑, 水果感, 温热感]
sem = {
    "芒果冰沙": np.array([0.95, 0.90, 0.0]),
    "杨枝甘露": np.array([0.80, 0.75, 0.0]),
    "西瓜冰":   np.array([0.98, 0.85, 0.0]),
    "热红豆沙": np.array([0.0, 0.0, 0.95]),
    "桂花酸梅汤": np.array([0.60, 0.0, 0.1]),
}
query_text = "解暑的甜品"
query_vec = np.array([0.9, 0.3, 0.0])   # "解暑"主要落在冰爽维度上

def cos(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

def jaccard(s1, s2):
    a, b = set(s1), set(s2)
    return len(a & b) / len(a | b) if a | b else 0.0

print(f"顾客搜索：「{query_text}」\n")
print(f"{'产品':　<8}{'关键词重合度':>10}{'语义余弦':>10}")
for name, v in sorted(sem.items(), key=lambda kv: -cos(kv[1], query_vec)):
    print(f"{name:　<8}{jaccard(query_text, name):>10.3f}{cos(query_vec, v):>10.3f}")

print("\n→ 关键词重合度几乎全 0（查询和产品名没有公共字符）；")
print("→ 语义余弦把三个冰饮排到前列：这就是 Embedding 的价值——'意思'进了向量，而不是'字'进了索引。")

## 实验 3：归一化之后，余弦 ≡ 欧氏（一个恒等式的数值验证）

课本结论：向量 L2 归一化后 `‖a−b‖² = 2(1−cos(a,b))`。
所以工程上"全部归一化 + 用欧氏距离索引"和"算余弦"完全等价——用随机矩阵验证到机器精度。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
A = rng.normal(size=(200, 32))
An = A / np.linalg.norm(A, axis=1, keepdims=True)

a, b = An[0], An[1]
d2_euclid = np.sum((a - b) ** 2)
d2_formula = 2 * (1 - a @ b)
print(f"直接算  ‖a-b‖²        = {d2_euclid:.15f}")
print(f"恒等式  2(1-cos)       = {d2_formula:.15f}")
print(f"误差                      = {abs(d2_euclid - d2_formula):.2e}  （机器精度级）")

# 全矩阵验证
D2 = ((An[:, None, :] - An[None, :, :]) ** 2).sum(-1)
C = An @ An.T
print(f"全矩阵最大误差 = {np.abs(D2 - 2 * (1 - C)).max():.2e}")
print("\n工程含义：先把所有向量归一化存进向量库，之后欧氏距离和余弦排序完全一致，")
print("         所以 FAISS/Milvus 里常见 'L2 索引 + 归一化向量' 的组合。")

## 实验 4：把两种"邻近"画出来 —— 方向锥 vs 半径圆

在 2 维语义平面上（横轴水果感、纵轴冰爽感）画出：
- **余弦检索**的邻域 = 查询方向两侧 ±40° 的扇形（只管方向，不管远近）；
- **欧氏检索**的邻域 = 查询周围的半径圆（方向和幅度一起管）。
注意"西瓜冰(小分量)"：方向几乎和查询一致（余弦≈1），却被欧氏圆排除在外。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Wedge, Circle

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

# 2 维语义平面：(水果感, 冰爽感)
pts = {
    "芒果冰沙": (0.90, 0.90),
    "杨枝甘露": (0.80, 0.70),
    "草莓圣代": (0.85, 0.30),
    "热红豆沙": (0.05, 0.05),
    "西瓜冰(小)": (0.30, 0.33),   # 方向与查询一致但幅度小
}
q = np.array([0.85, 0.95])

fig, ax = plt.subplots(figsize=(7, 6))
theta = np.degrees(np.arctan2(q[1], q[0]))
ax.add_patch(Wedge((0, 0), 1.35, theta - 40, theta + 40, alpha=0.15, color="#4f81bd"))
ax.add_patch(Circle(q, 0.30, fill=False, ls="--", color="#c0504d"))

for name, (x, y) in pts.items():
    in_cos = abs(np.degrees(np.arctan2(y, x)) - theta) <= 40
    ax.scatter([x], [y], s=90 if in_cos else 50, color="#4f81bd" if in_cos else "gray")
    ax.annotate(f"{name}\n{'余弦✓' if in_cos else '余弦✗'}", (x, y), xytext=(6, -14),
                textcoords="offset points", fontsize=9)
ax.scatter(*q, marker="*", s=300, color="#c0504d", zorder=5)
ax.annotate("查询意图", q, xytext=(8, 6), textcoords="offset points", color="#c0504d")

ax.set_xlim(0, 1.35); ax.set_ylim(0, 1.35)
ax.set_xlabel("水果感"); ax.set_ylabel("冰爽感")
ax.set_title("余弦=方向锥(蓝色扇形)，欧氏=半径圆(红色虚线)\n注意「西瓜冰(小)」：余弦✓但被欧氏圆排除")
plt.tight_layout(); plt.show()

print("结论：关心'说的是不是一回事'→ 余弦（方向）；关心'向量本身差多少'→ 欧氏。")
print("     Embedding 检索几乎都用余弦/归一化内积，因为文本'强度'（长度）不该左右语义排序。")

## 结论

| 问题 | 实验证据 |
|---|---|
| 三种度量何时打架 | 实验1：向量 x3 后余弦不变、欧氏/点积剧变，点积排序被幅度绑架 |
| 为什么要语义向量 | 实验2："解暑的甜品"与产品名零字符重合，语义余弦照样召回冰饮 |
| 余弦 vs 欧氏的关系 | 实验3：归一化后 ‖a−b‖²=2(1−cos)，误差 ~1e-16 |
| 怎么直观理解 | 实验4：余弦=方向锥，欧氏=半径圆，两者邻域形状根本不同 |

→ 深入阅读：`ima/第4周-Day2-向量检索与相似度计算.md`（Embedding 训练直觉、糖水店搜索场景、4 个误区）